# Step 02 — Agentic dbt Model Generator

**Goal:** A true LLM agent that autonomously queries the Neo4j knowledge graph and generates dbt models.

The agent is given **tools** and decides itself:
- What Cypher queries to run to gather context
- When it has enough information to generate the model
- What SQL and YAML to write

```
Orchestrator (Python)
  │  gives task: "generate model for `film`"
  ▼
LLM Agent  ──tool_call──▶  run_cypher_query()  ──▶  Neo4j
           ◀──result──────────────────────────────────────
           ──tool_call──▶  run_cypher_query()  (again if needed)
           ──tool_call──▶  write_dbt_model()   ──▶  disk
           ──tool_call──▶  write_schema_entry() ──▶  disk
  │  done — move to next object
```

**Stack:** `anthropic` (tool use) · `neo4j` · `networkx`

## Step 1 — Install dependencies

In [20]:
%pip install openai neo4j networkx python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Config

In [21]:
import os
import json
from pathlib import Path
from collections import defaultdict

from dotenv import load_dotenv
from openai import OpenAI
import networkx as nx
from neo4j import GraphDatabase

# Load .env from the same folder as this notebook
_env = Path(r"C:\Users\fdepoorter\OneDrive - Sopra Steria\Documents\dbt-migration-agent\.env")
load_dotenv(_env)

NEO4J_URI      = "neo4j://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "migration123"

GITHUB_TOKEN    = os.environ.get("GITHUB_TOKEN", "")
DBT_SOURCE_NAME = "legacy_mysql"
OUTPUT_DIR      = Path("dbt_output/models")
STAGING_DIR     = OUTPUT_DIR / "staging"
MARTS_DIR       = OUTPUT_DIR / "marts"
STAGING_DIR.mkdir(parents=True, exist_ok=True)
MARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory : {OUTPUT_DIR.resolve()}")
print(f"Token loaded     : {'yes' if GITHUB_TOKEN else 'NO — check .env'}")


Output directory : C:\Users\fdepoorter\OneDrive - Sopra Steria\Documents\dbt-migration-agent\dbt_output\models
Token loaded     : yes


## Step 3 — Connect and compute wave order

In [22]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


In [23]:
# Build a local graph just for topological ordering
G_topo = nx.DiGraph()

with driver.session() as session:
    for r in session.run("""
        MATCH (n) WHERE n.node_type IN ['table','view','procedure','referenced']
        RETURN n.node_id AS nid, n.node_type AS ntype
    """):
        G_topo.add_node(r["nid"], node_type=r["ntype"])

    for r in session.run("""
        MATCH (a)-[:FK|DEPENDS_ON|TOUCHES]->(b)
        WHERE a.node_type IN ['table','view','procedure']
          AND b.node_type IN ['table','view','procedure','referenced']
        RETURN DISTINCT a.node_id AS src, b.node_id AS dst
    """):
        G_topo.add_edge(r["src"], r["dst"])

tiers = {}
for node in nx.topological_sort(G_topo):
    deps = list(G_topo.successors(node))
    tiers[node] = 0 if not deps else max(tiers.get(d, 0) for d in deps) + 1

by_tier = defaultdict(list)
for node, tier in tiers.items():
    by_tier[tier].append(node)

for t in sorted(by_tier):
    print(f"  Wave {t}: {by_tier[t]}")

  Wave 0: ['film_text', 'customer', 'actor', 'category', 'film']
  Wave 1: ['actor_info', 'film_list', 'ins_film', 'list_active_customers', 'film_actor', 'film_category']


## Step 4 — Define agent tools

The LLM sees three tools and decides when and how to call them:
- `run_cypher_query` — query Neo4j for any context it needs
- `write_dbt_model` — write the generated `.sql` file to disk
- `write_schema_entry` — queue the `schema.yml` fragment

In [24]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_cypher_query",
            "description": (
                "Execute a read-only Cypher query against the Neo4j knowledge graph. "
                "Use this to look up node properties, columns, FK constraints, "
                "view dependencies, column lineage, or any other graph data you need."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Cypher query to execute"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_dbt_model",
            "description": "Write the final dbt SQL model to disk. Call once you have the complete SQL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "node_id": {"type": "string", "description": "Model name (used as filename)"},
                    "sql":     {"type": "string", "description": "Complete dbt SQL model content"}
                },
                "required": ["node_id", "sql"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_schema_entry",
            "description": "Queue the schema.yml fragment for this model. Call after write_dbt_model.",
            "parameters": {
                "type": "object",
                "properties": {
                    "node_id": {"type": "string"},
                    "yaml":    {"type": "string", "description": "schema.yml fragment for this model"}
                },
                "required": ["node_id", "yaml"]
            }
        }
    }
]

print(f"Defined {len(TOOLS)} tools")

Defined 3 tools


## Step 5 — Tool executor

Runs the actual tool when the LLM calls it.

In [25]:
schema_fragments = {"staging": [], "marts": []}

def execute_tool(tool_name, tool_input):
    if tool_name == "run_cypher_query":
        query = tool_input["query"]
        print(f"    ↳ Cypher: {query[:90]}{'...' if len(query) > 90 else ''}")
        with driver.session() as session:
            results = [dict(r) for r in session.run(query)]
        # Decode JSON-serialised lists stored by notebook 01
        for row in results:
            for k, v in row.items():
                if isinstance(v, str):
                    try:
                        row[k] = json.loads(v)
                    except (json.JSONDecodeError, ValueError):
                        pass
        return json.dumps(results, default=str)

    elif tool_name == "write_dbt_model":
        # node_id can be "staging/stg_film" or "marts/film"
        path = OUTPUT_DIR / f"{tool_input['node_id']}.sql"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(tool_input["sql"], encoding="utf-8")
        print(f"    ↳ wrote {path.relative_to(OUTPUT_DIR)}")
        return f"OK: {path}"

    elif tool_name == "write_schema_entry":
        node_id = tool_input["node_id"]
        layer = "staging" if node_id.startswith("staging/") else "marts"
        schema_fragments[layer].append(tool_input["yaml"])
        print(f"    ↳ schema entry queued for {node_id}")
        return "OK"

    return f"Unknown tool: {tool_name}"

print("Tool executor ready")


Tool executor ready


## Step 6 — System prompt

Gives the agent the graph schema and migration rules.

In [26]:
SYSTEM_PROMPT = f"""
You are a dbt model generator for a MySQL → Databricks migration.
You have access to a Neo4j knowledge graph via `run_cypher_query`.

## Graph schema
Node labels   : TABLE, VIEW, PROCEDURE, COLUMN, REFERENCED
Relationships : FK, DEPENDS_ON, TOUCHES, HAS_COLUMN, COL_LINEAGE

Key node properties:
  node_id   — the primary identifier for all nodes (e.g. 'film', 'actor')
              ALWAYS filter nodes by node_id, NOT by 'name'
  node_type — 'table', 'view', 'procedure', 'referenced'

Key COLUMN node properties:
  col_name, col_type, nullable, is_pk, auto_increment, default,
  mysql_type (ENUM/SET), accepted_values (list), table (parent node_id)

## Example Cypher patterns
  MATCH (n {{node_id: 'film'}}) RETURN n
  MATCH (n {{node_id: 'film'}})-[:HAS_COLUMN]->(c:COLUMN) RETURN c
  MATCH (n {{node_id: 'film'}})-[:FK]->(ref) RETURN ref
  MATCH (n {{node_id: 'actor_info'}})-[:DEPENDS_ON]->(dep) RETURN dep

## dbt project structure  (Jaffle Shop convention)
models/
  staging/    ← one model per source table, named stg_<table>
  marts/      ← business models: views, procedures, joined outputs

## Layer rules
- MySQL TABLE  → TWO models:
    1. staging/stg_<table_name>  — SELECT all columns from source with type casts
       Source ref: {{{{ source('{DBT_SOURCE_NAME}', '<table_name>') }}}}
    2. marts/<table_name>        — SELECT * FROM {{{{ ref('stg_<table_name>') }}}}
       (add business tests, descriptions here)

- MySQL VIEW   → marts/<view_name>
    Rewrite the SELECT logic using {{{{ ref('stg_<dep>') }}}} for each dependency
    assessed layer: intermediate if other objects consume it, mart if final output

- MySQL PROCEDURE → marts/<procedure_name>
    If it's a SELECT procedure: rewrite as a dbt SELECT model
    If it's an INSERT/UPDATE procedure: write a placeholder with TODO comment
    Use {{{{ ref() }}}} for any tables it touches

- REFERENCED  → marts/<name> with TODO placeholder (DDL unknown)

## Type cast rules (apply in staging models)
- UNSIGNED int        → CAST(col AS INT) or CAST(col AS BIGINT)
- YEAR type           → CAST(col AS INT)
- ENUM column         → CAST(col AS STRING); add accepted_values test in schema.yml
- SET column          → CAST(col AS STRING)
- CURRENT_TIMESTAMP   → current_timestamp()
- AUTO_INCREMENT PK   → keep as-is (source system identity)
- composite PK        → add dbt_utils.unique_combination_of_columns test
- FK column           → add relationships test in schema.yml

## Your task
1. Call `run_cypher_query` to fetch columns, FKs, and dependencies for the node
2. For TABLE nodes: generate BOTH staging/stg_<name> AND marts/<name>
   — use write_dbt_model twice (once per file)
   — use write_schema_entry twice (once per file)
3. For VIEW/PROCEDURE nodes: generate only the marts/<name> model
4. Always use explicit column names — never SELECT *
""".strip()

print("System prompt ready")


System prompt ready


## Step 7 — Agent function

The agentic loop: keeps running until `end_turn` (agent has called write tools and is done).

In [27]:
llm_client = OpenAI(
    base_url="https://api.githubcopilot.com",
    api_key=GITHUB_TOKEN,
)

def run_agent(node_id, node_type, wave):
    print(f"\n  [{node_type.upper():12s}] {node_id}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Generate a dbt model for `{node_id}` "
                f"(MySQL type: {node_type}, migration wave: {wave}).\n"
                "Query the graph for all context you need, then write the SQL and schema entry."
            )
        }
    ]

    while True:
        response = llm_client.chat.completions.create(
            model="gpt-4o",
            tools=TOOLS,
            messages=messages,
        )

        choice = response.choices[0]
        msg    = choice.message

        # Append assistant turn
        assistant_msg = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            assistant_msg["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(assistant_msg)

        if choice.finish_reason == "stop":
            break

        if choice.finish_reason == "tool_calls":
            for tc in msg.tool_calls:
                result = execute_tool(tc.function.name, json.loads(tc.function.arguments))
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
        else:
            break

print("Agent ready")

Agent ready


## Step 8 — Run the agent wave by wave

In [ ]:
import shutil

# Clean previous output so stale files don't accumulate
for d in [STAGING_DIR, MARTS_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

schema_fragments["staging"].clear()
schema_fragments["marts"].clear()

for wave in sorted(by_tier.keys()):
    print(f"\n{'='*60}")
    print(f"WAVE {wave}")
    print(f"{'='*60}")
    for node_id in by_tier[wave]:
        node_type = G_topo.nodes[node_id].get("node_type", "unknown")
        run_agent(node_id, node_type, wave)



WAVE 0

  [REFERENCED  ] film_text
    ↳ Cypher: MATCH (n {node_id: 'film_text'}) RETURN n
    ↳ wrote film_text.sql
    ↳ schema entry queued for film_text

  [TABLE       ] customer
    ↳ Cypher: MATCH (n {node_id: 'customer'})-[:HAS_COLUMN]->(c:COLUMN) RETURN c
    ↳ Cypher: MATCH (n {node_id: 'customer'})-[:FK]->(ref) RETURN ref
    ↳ wrote stg_customer.sql
    ↳ schema entry queued for stg_customer
    ↳ wrote customer.sql
    ↳ schema entry queued for customer

  [TABLE       ] actor
    ↳ Cypher: MATCH (n {node_id: 'actor'})-[:HAS_COLUMN]->(c:COLUMN) RETURN c
    ↳ Cypher: MATCH (n {node_id: 'actor'})-[:FK]->(ref) RETURN ref
    ↳ wrote stg_actor.sql
    ↳ schema entry queued for stg_actor
    ↳ wrote actor.sql
    ↳ schema entry queued for actor

  [TABLE       ] category
    ↳ Cypher: MATCH (n {node_id: 'category'})-[:HAS_COLUMN]->(c:COLUMN) RETURN c
    ↳ Cypher: MATCH (n {node_id: 'category'})-[:FK]->(ref) RETURN ref
    ↳ wrote stg_category.sql
    ↳ schema entry queued fo

In [29]:
# Flush separate schema.yml per layer
for layer, frags in schema_fragments.items():
    if not frags:
        continue
    schema_path = OUTPUT_DIR / layer / "schema.yml"
    combined = "version: 2\n\nmodels:\n"
    for frag in frags:
        for line in frag.splitlines():
            combined += f"  {line}\n"
        combined += "\n"
    schema_path.write_text(combined, encoding="utf-8")
    print(f"schema.yml written → {schema_path}")


schema.yml written → dbt_output\models\marts\schema.yml


## Step 9 — Review generated output

In [30]:
for f in sorted(OUTPUT_DIR.rglob("*.sql")):
    rel = f.relative_to(OUTPUT_DIR)
    print(f"\n{'='*40}  {rel}  {'='*40}")
    print(f.read_text(encoding="utf-8"))



========================================  actor.sql  ========================================
SELECT 
    actor_id,
    first_name,
    last_name,
    last_update
FROM {{ ref('stg_actor') }}

========================================  actor_info.sql  ========================================
-- Models the `actor_info` view in Databricks.

WITH actor AS (
  SELECT 
    CAST(actor_id AS INT) AS actor_id,
    first_name,
    last_name,
    last_update
  FROM {{ ref('stg_actor') }}
),
film_actor AS (
  SELECT 
    CAST(actor_id AS INT) AS actor_id,
    CAST(film_id AS INT) AS film_id,
    last_update
  FROM {{ ref('stg_film_actor') }}
),
film_category AS (
  SELECT 
    CAST(film_id AS INT) AS film_id,
    CAST(category_id AS INT) AS category_id,
    last_update
  FROM {{ ref('stg_film_category') }}
),
category AS (
  SELECT 
    CAST(category_id AS INT) AS category_id,
    name AS category_name,
    last_update
  FROM {{ ref('stg_category') }}
)

SELECT 
  a.actor_id,
  a.first_name,
  a.l

In [31]:
print((OUTPUT_DIR / "schema.yml").read_text(encoding="utf-8"))

FileNotFoundError: [Errno 2] No such file or directory: 'dbt_output\\models\\schema.yml'